In [1]:
# Import Necessary Packages/Modules
from skeLCS import eLCS # Run "pip install scikit-eLCS" to install necessary module
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score

In [2]:
# Read file into a table. Turns off default behavior that reads "none" as missing
df = pd.read_csv("medical_insurance_cleaned.csv", keep_default_na=False, na_values=[""])

In [3]:
# eLCS can only predict classes, so "risk_score" will be cut into 3 groups
df["risk_band"] = pd.qcut(df["risk_score"], 3, labels=False) # labels=False gives plain integers, which are better for machine learning
y = df["risk_band"].values # converts the column to a NumPy array

In [4]:
# Define what the leaky columns are
LEAKAGE_COLS = ["is_high_risk", "annual_premium", "monthly_premium", "annual_medical_cost", "claims_count", "avg_claim_amount", "total_claims_paid",]

# Excludes the ones columns that come as a result of "risk_score" along with the IDs
X = df.drop(columns=LEAKAGE_COLS + ["risk_score", "risk_band", "person_id"])

In [5]:
# eLCS needs numbers, so every text column becomes integer codes
for col in X.select_dtypes(exclude="number"):
    X[col] = X[col].astype("category").cat.codes

In [6]:
# Converts the table to a plain array of decimals
X = X.values.astype(float)

In [7]:
%%time
# Run and score model
model = eLCS(random_state=42) # Seeded model with a state to make results repeatable
cv = StratifiedKFold(3, shuffle=True, random_state=42) # Sets up 3-fold cross validation
baseline_scores = cross_val_score(model, X, y, cv=cv, scoring="balanced_accuracy") # Runs the test loop 3 times and returns 3 scores with the scoring set to balanced accuracy
print(f"Baseline eLCS (45 features), CV balanced accuracy: {baseline_scores.mean():.3f}")
print(f"Fold scores: {np.round(baseline_scores, 3)}")

Baseline eLCS (45 features), CV balanced accuracy: 0.568
Fold scores: [0.505 0.641 0.56 ]
CPU times: total: 2min 59s
Wall time: 3min 30s


In [8]:
%%time
# Fresh model trained on entire dataset
model = eLCS(random_state=42)
model.fit(X, y)
print(f"Training accuracy: {model.get_final_accuracy():.3f}") # How accurate is is the final set of rules
print(f"Instance coverage: {model.get_final_instance_coverage():.1%}") # Percentage of training rows covered by at least one rule

Training accuracy: 0.488
Instance coverage: 74.5%
CPU times: total: 3min 41s
Wall time: 4min 41s


In [9]:
# Comparison with decision tree
tree = DecisionTreeClassifier(max_depth=5, random_state=42) # Shallow decision tree, kept seeded for consistency
tree_scores = cross_val_score(tree, X, y, cv=cv, scoring="balanced_accuracy") # 3-fold cross validation with the scoring set to balanced accuracy
print(f"Decision tree reference, CV balanced accuracy: {tree_scores.mean():.3f}")

Decision tree reference, CV balanced accuracy: 0.953


In [10]:
#Rerunning previous parts of code to get the labels back for a check
df = pd.read_csv("medical_insurance.csv", keep_default_na=False, na_values=[""])

df["risk_band"] = pd.qcut(df["risk_score"], 3, labels=False)
y = df["risk_band"].values

LEAKAGE_COLS = ["is_high_risk", "annual_premium", "monthly_premium", "annual_medical_cost", "claims_count", "avg_claim_amount", "total_claims_paid",]

X = df.drop(columns=LEAKAGE_COLS + ["risk_score", "risk_band", "person_id"])

for col in X.select_dtypes(exclude="number"):
    X[col] = X[col].astype("category").cat.codes

# Checking for top features
feature_names = X.columns   # Get the column names of the features
tree.fit(X, y)  # Train a decision tree on the full dataset
print(pd.Series(tree.feature_importances_, index=feature_names).sort_values(ascending=False).head(10))  # feature_importances_ gives one number per feature that add up to 1

# Converts the table back into to a plain array of decimals for future use
X = X.values.astype(float)

chronic_count     0.404078
age               0.366792
smoker            0.160757
bmi               0.068372
sex               0.000000
income            0.000000
urban_rural       0.000000
education         0.000000
marital_status    0.000000
household_size    0.000000
dtype: float64


In [11]:
%%time
# eLCS with only the top 4 features
SELECTED = ["chronic_count", "age", "smoker", "bmi"]

# Builds a new table with just those columns, then encodes them
X = df[SELECTED].copy()
for col in X.select_dtypes(exclude="number"):
    X[col] = X[col].astype("category").cat.codes
X = X.values.astype(float)

model = eLCS(random_state=42)
cv = StratifiedKFold(3, shuffle=True, random_state=42)
selected_scores = cross_val_score(model, X, y, cv=cv)
print(f"eLCS (4 features), CV balanced accuracy: {selected_scores.mean():.3f}")
print(f"Fold scores: {np.round(selected_scores, 3)}")

eLCS (4 features), CV balanced accuracy: 0.827
Fold scores: [0.821 0.862 0.799]
CPU times: total: 1min 38s
Wall time: 2min 18s


In [12]:
# Printing the parameters used for our tests
print(eLCS(random_state=42).get_params())

{'N': 1000, 'acc_sub': 0.99, 'beta': 0.2, 'chi': 0.8, 'delta': 0.1, 'discrete_attribute_limit': 10, 'do_GA_subsumption': True, 'do_correct_set_subsumption': False, 'fitness_reduction': 0.1, 'init_fit': 0.01, 'learning_iterations': 10000, 'match_for_missingness': False, 'mu': 0.04, 'nu': 5, 'p_spec': 0.5, 'random_state': 42, 'reboot_filename': None, 'selection_method': 'tournament', 'specified_attributes': array([], dtype=float64), 'theta_GA': 25, 'theta_del': 20, 'theta_sel': 0.5, 'theta_sub': 20, 'track_accuracy_while_fit': False}


In [13]:
results = pd.DataFrame({
    "Model": ["eLCS baseline (45 features)", "Decision tree (reference)", "eLCS (4 features)"],
    "CV balanced accuracy": [baseline_scores.mean(), tree_scores.mean(), selected_scores.mean()],
}).round(3)
results

,Model,CV balanced accuracy
0,eLCS baseline (45 features),0.568
1,Decision tree (reference),0.953
2,eLCS (4 features),0.827
